# LimX HU_D03: locomotion and mimic training with mjlab (Colab / Kaggle)

Trains two policies for the LimX Dynamics HU_D03 humanoid with
[mjlab](https://github.com/mujocolab/mjlab) on a free cloud GPU:

1. **Locomotion** (`Mjlab-Velocity-Flat-LimX-HU-D03`): follow forward/sideways/turn velocity commands.
2. **Mimic** (`Mjlab-Tracking-Flat-LimX-HU-D03`): track a reference motion from LAFAN1.

**Pick a GPU runtime first.** mjlab needs an NVIDIA GPU of the Turing generation or newer.

| | Colab | Kaggle |
|---|---|---|
| Runtime | *Runtime → Change runtime type → T4 GPU* | *Settings → Accelerator → GPU T4 x2* (**not P100**: too old for mjlab) |
| Internet | on by default | *Settings → Internet → On* (needs a phone-verified account) |
| Limits | about 12 h per session at most, often less; disconnects when idle | 12 h per session, about 30 GPU h per week |
| Where results are kept | your Google Drive (mounted below) | `/kaggle/working` (kept when you *Save Version*) |

Sessions end without warning, so checkpoints are saved every 50 (locomotion) or 100 (mimic) iterations
and every training cell can **resume** from the latest checkpoint.

**About the HU_D03 model.** The ankles and waist on the real robot are parallel linkages (two motors per joint pair).
This notebook replaces them with serial `ankle_pitch/roll` and `waist_roll/pitch` joints, as mjlab does for the
Unitree G1, so the policy outputs serial joint targets. On the real robot these must be converted to the A/B motor
commands. PD gains are not published by LimX and are derived from the rotor inertias (10 Hz, damping ratio 2).

## 1. Settings

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if ON_COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  WORK = Path("/content/drive/MyDrive/hu_d03_rl")  # Logs and checkpoints survive disconnects.
elif ON_KAGGLE:
  WORK = Path("/kaggle/working/hu_d03_rl")
else:
  WORK = Path.home() / "hu_d03_rl"
CODE = Path("/tmp/hu_d03_code")  # Installed code (fast local disk, re-created each session).
WORK.mkdir(parents=True, exist_ok=True)
CODE.mkdir(parents=True, exist_ok=True)

# Training sizes. 4096 envs is what the PPO settings are tuned for; use 2048 if you run out of GPU memory.
NUM_ENVS = 4096
VELOCITY_ITERATIONS = 3000   # Raise and resume if the robot is not walking well yet.
TRACKING_ITERATIONS = 5000
USE_TWO_GPUS = ON_KAGGLE     # Kaggle "GPU T4 x2" gives two GPUs.

os.environ["WANDB_MODE"] = "disabled"   # Log to TensorBoard instead of Weights & Biases.
os.environ["MUJOCO_GL"] = "egl"
os.environ["MJLAB_WARP_QUIET"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"  # Stream training logs into the notebook.
os.environ["PYTHONFAULTHANDLER"] = "1"  # Print a Python traceback on a crash.
os.environ["HU_D03_DESCRIPTION_DIR"] = str(CODE / "humanoid-description" / "HU_D03_description")
print("work dir:", WORK)

## 2. Check the GPU

In [ ]:
out = subprocess.run(
  ["nvidia-smi", "--query-gpu=name,compute_cap,driver_version,memory.total", "--format=csv,noheader"],
  capture_output=True, text=True,
)
if out.returncode != 0:
  raise SystemExit("No NVIDIA GPU found. Switch the runtime to a T4 GPU (see the table above).")
print(out.stdout)
name, cap, driver, _ = [s.strip() for s in out.stdout.splitlines()[0].split(",")]
if float(cap) < 7.5:
  raise SystemExit(f"{name} (compute {cap}) is older than Turing; mjlab needs a T4 or newer.")
DRIVER_MAJOR = int(driver.split(".")[0])
# mjlab's CUDA 13 build needs driver 580+; older drivers use its cu126 extra.
CUDA_EXTRA = "cu130" if DRIVER_MAJOR >= 580 else "cu126"
NUM_GPUS = len(out.stdout.splitlines())
print("mjlab extra:", CUDA_EXTRA, "| GPUs:", NUM_GPUS)

## 3. Install mjlab and the LimX HU_D03 model (about 3–5 minutes)

In [ ]:
MJLAB_COMMIT = "8bb2bc9e187797689ec82c78c35019afd2fd0376"
DESCRIPTION_COMMIT = "0dd8454384c58991f3e70fd7613be29006d39f86"

def sh(cmd):
  # Run a shell command, streaming its output into the notebook.
  print("$", cmd, flush=True)
  proc = subprocess.Popen(cmd, shell=True, text=True, bufsize=1,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  for line in proc.stdout:
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise SystemExit(f"command failed with exit code {proc.returncode}")

def fetch(repo, commit, dest):
  if not (dest / ".git").exists():
    sh(f"git init -q {dest} && git -C {dest} remote add origin {repo}")
  sh(f"git -C {dest} fetch -q --depth 1 origin {commit} && git -C {dest} checkout -q FETCH_HEAD")

fetch("https://github.com/mujocolab/mjlab.git", MJLAB_COMMIT, CODE / "mjlab")
fetch("https://github.com/limxdynamics/humanoid-description.git", DESCRIPTION_COMMIT, CODE / "humanoid-description")

sh("pip install -q uv")
sh(f"uv pip install --system -q -e '{CODE / 'mjlab'}[{CUDA_EXTRA}]'")

### Write the HU_D03 package

These cells write the `hu_d03_mjlab` package (robot model and task configs) and the helper scripts.
They are generated from the `hu-d03-rl` folder; edit them there and re-run `notebooks/build_notebook.py`,
or edit them here directly.

In [ ]:
for d in ["src/hu_d03_mjlab/robot", "src/hu_d03_mjlab/tasks", "scripts"]:
  (CODE / "hu-d03-rl" / d).mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/pyproject.toml
[project]
name = "hu-d03-mjlab"
version = "0.1.0"
description = "LimX HU_D03 locomotion and motion-tracking tasks for mjlab"
requires-python = ">=3.10"
dependencies = ["mjlab"]

[project.entry-points."mjlab.tasks"]
hu_d03 = "hu_d03_mjlab.tasks"

[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[tool.setuptools.packages.find]
where = ["src"]

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/__init__.py
"""LimX HU_D03 tasks for mjlab."""

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/__init__.py
"""LimX HU_D03 robot model for mjlab."""

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_spec.py
"""Load the LimX HU_D03 MJCF and turn it into a serial-joint model for mjlab.

This module only depends on `mujoco`, so it can be tested on a CPU-only
machine without mjlab installed.

What the conversion does:
  * Removes the parallel ankle (A/B achilles) and waist (A/B) linkages and the 6
    `<connect>` constraints that close them. The policy then commands the
    serial joints `ankle_pitch/roll` and `waist_roll/pitch` directly, like the
    Unitree G1 model in mjlab. The linkage mass is added to the parent body.
  * Removes the floor, light, camera, actuators and sensors from the source XML
    (mjlab adds its own terrain, actuators and sensors).
  * Sets every geom margin to 0 (MuJoCo Warp does not support margins on
    box/mesh pairs, such as the two foot boxes).
  * Names every collision geom (`<body>_collision<i>`, feet are
    `left_foot_collision` / `right_foot_collision`) so mjlab can match them.
  * Adds foot sites at the sole and the IMU sensors mjlab's tasks read.
"""

import os
from pathlib import Path

import mujoco
import numpy as np

_DEFAULT_DESCRIPTION_DIR = (
  Path(__file__).resolve().parents[3]
  / "third_party"
  / "humanoid-description"
  / "HU_D03_description"
)


def description_dir() -> Path:
  """Directory of `HU_D03_description` (override with $HU_D03_DESCRIPTION_DIR)."""
  return Path(os.environ.get("HU_D03_DESCRIPTION_DIR", _DEFAULT_DESCRIPTION_DIR))


def source_xml() -> Path:
  xml = description_dir() / "xml" / "HU_D03_03.xml"
  if not xml.exists():
    raise FileNotFoundError(
      f"{xml} not found. Clone https://github.com/limxdynamics/humanoid-description "
      "and set HU_D03_DESCRIPTION_DIR to its HU_D03_description folder."
    )
  return xml


# Bodies (with their subtrees) that form the closed-loop linkages.
LINKAGE_BODIES = (
  "left_A_achilles_link",
  "left_B_achilles_link",
  "right_A_achilles_link",
  "right_B_achilles_link",
  "waist_A_link",
  "waist_B_link",
)

# Foot sole: the foot box is centred 0.0535 m below the ankle roll frame and is
# 0.01 m half-thick, so the sole is at z = -0.0635.
FOOT_SITE_POS = (0.018, 0.0, -0.0635)
FOOT_GEOMS = {"left_foot": "left_foot_collision", "right_foot": "right_foot_collision"}


def _subtree_mass(body: mujoco.MjsBody) -> float:
  mass = body.mass
  for child in body.bodies:
    mass += _subtree_mass(child)
  return mass


def _add_mass(body: mujoco.MjsBody, extra: float) -> None:
  """Add point-like mass to a body, scaling its inertia by the same ratio."""
  ratio = (body.mass + extra) / body.mass
  body.mass = body.mass + extra
  body.inertia = np.asarray(body.inertia) * ratio


def get_spec() -> mujoco.MjSpec:
  spec = mujoco.MjSpec.from_file(str(source_xml()))

  # Equality constraints, actuators and sensors all reference linkage joints.
  for eq in list(spec.equalities):
    spec.delete(eq)
  for act in list(spec.actuators):
    spec.delete(act)
  for sensor in list(spec.sensors):
    spec.delete(sensor)

  # Floor, light and tracking camera live in the worldbody; mjlab adds terrain.
  for geom in list(spec.worldbody.geoms):
    spec.delete(geom)
  for light in list(spec.worldbody.lights):
    spec.delete(light)
  for cam in list(spec.worldbody.cameras):
    spec.delete(cam)

  # Remove the linkages, moving their mass to the parent body.
  for name in LINKAGE_BODIES:
    body = spec.body(name)
    parent = body.parent
    _add_mass(parent, _subtree_mass(body))
    spec.delete(body)

  # The free joint has a tiny spring/damper in the source XML; drop them.
  root = spec.body("base_link")
  free = root.joints[0]
  free.damping = np.zeros_like(free.damping)
  free.stiffness = np.zeros_like(free.stiffness)

  # Name collision geoms so mjlab's regex-based configs can find them.
  for body in spec.bodies:
    i = 0
    for geom in body.geoms:
      # The XML default gives every geom margin=0.001. MuJoCo Warp rejects a
      # non-zero margin on box/mesh pairs (e.g. foot-foot) with MULTICCD.
      geom.margin = 0.0
      if geom.contype == 0 and geom.conaffinity == 0:
        continue  # Visual mesh.
      if geom.name in FOOT_GEOMS:
        geom.name = FOOT_GEOMS[geom.name]
      elif not geom.name:
        geom.name = f"{body.name}_collision{i}"
        i += 1

  for side in ("left", "right"):
    spec.body(f"{side}_ankle_roll_link").add_site(
      name=f"{side}_foot", pos=FOOT_SITE_POS, size=(0.01, 0.01, 0.01), group=5
    )

  # Sensors read by mjlab's velocity and tracking tasks.
  spec.add_sensor(name="imu_ang_vel", type=mujoco.mjtSensor.mjSENS_GYRO,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_lin_vel", type=mujoco.mjtSensor.mjSENS_VELOCIMETER,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_lin_acc", type=mujoco.mjtSensor.mjSENS_ACCELEROMETER,
                  objtype=mujoco.mjtObj.mjOBJ_SITE, objname="imu")
  spec.add_sensor(name="imu_upvector", type=mujoco.mjtSensor.mjSENS_FRAMEZAXIS,
                  objtype=mujoco.mjtObj.mjOBJ_BODY, objname="world",
                  reftype=mujoco.mjtObj.mjOBJ_SITE, refname="imu")
  spec.add_sensor(name="root_angmom", type=mujoco.mjtSensor.mjSENS_SUBTREEANGMOM,
                  objtype=mujoco.mjtObj.mjOBJ_BODY, objname="base_link")

  return spec

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_params.py
"""HU_D03 actuator and pose parameters (plain Python, no mjlab import).

Armatures are the reflected rotor inertias from the LimX SRDF
(`rotor mass * gear_ratio**2`), which match the `armature` values in the MJCF.
Effort limits are the `ctrlrange` of the MJCF motors.

LimX does not publish the PD gains used on the real robot, so stiffness and
damping follow mjlab's G1 rule: a second-order system with natural frequency
NATURAL_FREQ_HZ and damping ratio DAMPING_RATIO for the joint's armature.
Replace them with the real gains if you get them from LimX.

The ankle and waist roll/pitch joints are driven by two motors through a
parallel linkage. As for the G1, we assume a nominal 1:1 linkage, so these
joints get the armature and effort of both motors.
"""

import math
from dataclasses import dataclass

NATURAL_FREQ_HZ = 10.0
DAMPING_RATIO = 2.0

ARMATURE_LEG = 0.000244114 * 25.0**2  # 0.1526, hips and knees.
ARMATURE_36 = 0.000073217 * 36.0**2  # 0.0949, ankle/waist motors.
ARMATURE_ARM = 0.000073217 * 25.0**2  # 0.0458, shoulders and elbows.
ARMATURE_SMALL = 0.000017 * 25.0**2  # 0.0106, wrists, hands and head.


@dataclass(frozen=True)
class ActuatorGroup:
  joint_names_expr: tuple[str, ...]
  armature: float
  effort_limit: float

  @property
  def stiffness(self) -> float:
    w = 2.0 * math.pi * NATURAL_FREQ_HZ
    return self.armature * w**2

  @property
  def damping(self) -> float:
    w = 2.0 * math.pi * NATURAL_FREQ_HZ
    return 2.0 * DAMPING_RATIO * self.armature * w


ACTUATOR_GROUPS = {
  "leg": ActuatorGroup(
    (".*_hip_pitch_joint", ".*_hip_roll_joint", ".*_hip_yaw_joint", ".*_knee_joint"),
    ARMATURE_LEG,
    120.0,
  ),
  "ankle": ActuatorGroup(
    (".*_ankle_pitch_joint", ".*_ankle_roll_joint"), 2 * ARMATURE_36, 2 * 45.0
  ),
  "waist_yaw": ActuatorGroup(("waist_yaw_joint",), ARMATURE_36, 45.0),
  "waist": ActuatorGroup(
    ("waist_roll_joint", "waist_pitch_joint"), 2 * ARMATURE_36, 2 * 45.0
  ),
  "arm": ActuatorGroup(
    (
      ".*_shoulder_pitch_joint",
      ".*_shoulder_roll_joint",
      ".*_shoulder_yaw_joint",
      ".*_elbow_joint",
    ),
    ARMATURE_ARM,
    30.0,
  ),
  "small": ActuatorGroup(
    (".*_wrist_yaw_joint", ".*_wrist_pitch_joint", ".*_hand_yaw_joint", "head_.*"),
    ARMATURE_SMALL,
    18.0,
  ),
}

# Standing pose with slightly bent knees. Signs: negative hip pitch swings the
# thigh forward, positive knee flexes, negative elbow flexes. The base height
# puts the soles on the ground (checked by scripts/stand_test.py).
HOME_BASE_HEIGHT = 0.889
HOME_JOINT_POS = {
  ".*_hip_pitch_joint": -0.25,
  ".*_knee_joint": 0.5,
  ".*_ankle_pitch_joint": -0.25,
  "left_shoulder_roll_joint": 0.15,
  "right_shoulder_roll_joint": -0.15,
  ".*_elbow_joint": -0.3,
}

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/robot/hu_d03_constants.py
"""LimX HU_D03 robot config for mjlab."""

from mjlab.actuator import BuiltinPositionActuatorCfg
from mjlab.entity import EntityArticulationInfoCfg, EntityCfg
from mjlab.utils.spec_config import CollisionCfg

from hu_d03_mjlab.robot.hu_d03_params import (
  ACTUATOR_GROUPS,
  HOME_BASE_HEIGHT,
  HOME_JOINT_POS,
)
from hu_d03_mjlab.robot.hu_d03_spec import get_spec

##
# Actuator config.
##

HU_D03_ACTUATORS = tuple(
  BuiltinPositionActuatorCfg(
    target_names_expr=g.joint_names_expr,
    stiffness=g.stiffness,
    damping=g.damping,
    effort_limit=g.effort_limit,
    armature=g.armature,
  )
  for g in ACTUATOR_GROUPS.values()
)

##
# Keyframe config.
##

HOME_KEYFRAME = EntityCfg.InitialStateCfg(
  pos=(0, 0, HOME_BASE_HEIGHT),
  joint_pos=HOME_JOINT_POS,
  joint_vel={".*": 0.0},
)

##
# Collision config.
##

FOOT_GEOMS = r"^(left|right)_foot_collision$"

# All collisions, including self collisions. Feet get condim=3, the rest condim=1.
FULL_COLLISION = CollisionCfg(
  geom_names_expr=(".*_collision.*",),
  contype=1,
  conaffinity=1,
  condim={FOOT_GEOMS: 3, ".*_collision.*": 1},
  priority={FOOT_GEOMS: 1, ".*": 0},
  friction={FOOT_GEOMS: (0.6,)},
)

##
# Final config.
##

HU_D03_ARTICULATION = EntityArticulationInfoCfg(
  actuators=HU_D03_ACTUATORS,
  soft_joint_pos_limit_factor=0.9,
)


def get_hu_d03_robot_cfg() -> EntityCfg:
  """Get a fresh HU_D03 robot configuration instance."""
  return EntityCfg(
    init_state=HOME_KEYFRAME,
    collisions=(FULL_COLLISION,),
    spec_fn=get_spec,
    articulation=HU_D03_ARTICULATION,
  )


# With the 10 Hz gains, 0.25 * effort / stiffness (the G1 rule) gives only
# 0.03-0.05 rad per unit action on the legs, because the HU_D03's reflected
# inertias are large. Use a flat 0.25 rad instead.
HU_D03_ACTION_SCALE: dict[str, float] = {".*": 0.25}

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/__init__.py
"""Registers the HU_D03 tasks with mjlab (loaded via the `mjlab.tasks` entry point)."""

from mjlab.tasks.registry import register_mjlab_task
from mjlab.tasks.tracking.rl import MotionTrackingOnPolicyRunner
from mjlab.tasks.velocity.rl import VelocityOnPolicyRunner

from .rl_cfgs import hu_d03_tracking_ppo_runner_cfg, hu_d03_velocity_ppo_runner_cfg
from .tracking_env_cfgs import hu_d03_flat_tracking_env_cfg
from .velocity_env_cfgs import hu_d03_flat_env_cfg, hu_d03_rough_env_cfg

register_mjlab_task(
  task_id="Mjlab-Velocity-Flat-LimX-HU-D03",
  env_cfg=hu_d03_flat_env_cfg(),
  play_env_cfg=hu_d03_flat_env_cfg(play=True),
  rl_cfg=hu_d03_velocity_ppo_runner_cfg(),
  runner_cls=VelocityOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Velocity-Rough-LimX-HU-D03",
  env_cfg=hu_d03_rough_env_cfg(),
  play_env_cfg=hu_d03_rough_env_cfg(play=True),
  rl_cfg=hu_d03_velocity_ppo_runner_cfg(),
  runner_cls=VelocityOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Tracking-Flat-LimX-HU-D03",
  env_cfg=hu_d03_flat_tracking_env_cfg(),
  play_env_cfg=hu_d03_flat_tracking_env_cfg(play=True),
  rl_cfg=hu_d03_tracking_ppo_runner_cfg(),
  runner_cls=MotionTrackingOnPolicyRunner,
)

register_mjlab_task(
  task_id="Mjlab-Tracking-Flat-LimX-HU-D03-No-State-Estimation",
  env_cfg=hu_d03_flat_tracking_env_cfg(has_state_estimation=False),
  play_env_cfg=hu_d03_flat_tracking_env_cfg(has_state_estimation=False, play=True),
  rl_cfg=hu_d03_tracking_ppo_runner_cfg(),
  runner_cls=MotionTrackingOnPolicyRunner,
)

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/rl_cfgs.py
"""PPO runner configs for the HU_D03 tasks (G1 hyperparameters)."""

from mjlab.rl import (
  RslRlModelCfg,
  RslRlOnPolicyRunnerCfg,
  RslRlPpoAlgorithmCfg,
)


def _runner_cfg(
  experiment_name: str, entropy_coef: float, save_interval: int
) -> RslRlOnPolicyRunnerCfg:
  return RslRlOnPolicyRunnerCfg(
    actor=RslRlModelCfg(
      hidden_dims=(512, 256, 128),
      activation="elu",
      obs_normalization=True,
      distribution_cfg={
        "class_name": "GaussianDistribution",
        "init_std": 1.0,
        "std_type": "scalar",
      },
    ),
    critic=RslRlModelCfg(
      hidden_dims=(512, 256, 128),
      activation="elu",
      obs_normalization=True,
    ),
    algorithm=RslRlPpoAlgorithmCfg(
      value_loss_coef=1.0,
      use_clipped_value_loss=True,
      clip_param=0.2,
      entropy_coef=entropy_coef,
      num_learning_epochs=5,
      num_mini_batches=4,
      learning_rate=1.0e-3,
      schedule="adaptive",
      gamma=0.99,
      lam=0.95,
      desired_kl=0.01,
      max_grad_norm=1.0,
    ),
    experiment_name=experiment_name,
    save_interval=save_interval,
    num_steps_per_env=24,
    max_iterations=30_000,
  )


def hu_d03_velocity_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  return _runner_cfg("hu_d03_velocity", entropy_coef=0.01, save_interval=50)


def hu_d03_tracking_ppo_runner_cfg() -> RslRlOnPolicyRunnerCfg:
  return _runner_cfg("hu_d03_tracking", entropy_coef=0.005, save_interval=100)

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/velocity_env_cfgs.py
"""LimX HU_D03 velocity environment configurations (adapted from mjlab's G1 ones)."""

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs import mdp as envs_mdp
from mjlab.envs.mdp.actions import JointPositionActionCfg
from mjlab.managers.event_manager import EventTermCfg
from mjlab.managers.reward_manager import RewardTermCfg
from mjlab.sensor import (
  ContactMatch,
  ContactSensorCfg,
  ObjRef,
  RayCastSensorCfg,
  RingPatternCfg,
  TerrainHeightSensorCfg,
)
from mjlab.tasks.velocity import mdp
from mjlab.tasks.velocity.mdp import UniformVelocityCommandCfg
from mjlab.tasks.velocity.velocity_env_cfg import make_velocity_env_cfg

from hu_d03_mjlab.robot.hu_d03_constants import (
  FOOT_GEOMS,
  HU_D03_ACTION_SCALE,
  get_hu_d03_robot_cfg,
)

PELVIS = "base_link"
TORSO = "waist_pitch_link"


def hu_d03_rough_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 rough terrain velocity configuration."""
  cfg = make_velocity_env_cfg()

  cfg.sim.mujoco.ccd_iterations = 500
  cfg.sim.contact_sensor_maxmatch = 500
  cfg.sim.nconmax = 70

  cfg.scene.entities = {"robot": get_hu_d03_robot_cfg()}

  for sensor in cfg.scene.sensors or ():
    if sensor.name == "terrain_scan":
      assert isinstance(sensor, RayCastSensorCfg)
      assert isinstance(sensor.frame, ObjRef)
      sensor.frame.name = PELVIS

  site_names = ("left_foot", "right_foot")

  for sensor in cfg.scene.sensors or ():
    if sensor.name == "foot_height_scan":
      assert isinstance(sensor, TerrainHeightSensorCfg)
      sensor.frame = tuple(
        ObjRef(type="site", name=s, entity="robot") for s in site_names
      )
      sensor.pattern = RingPatternCfg.single_ring(radius=0.03, num_samples=6)

  feet_ground_cfg = ContactSensorCfg(
    name="feet_ground_contact",
    primary=ContactMatch(
      mode="subtree",
      pattern=r"^(left_ankle_roll_link|right_ankle_roll_link)$",
      entity="robot",
    ),
    secondary=ContactMatch(mode="body", pattern="terrain"),
    fields=("found", "force"),
    reduce="netforce",
    num_slots=1,
    track_air_time=True,
  )
  self_collision_cfg = ContactSensorCfg(
    name="self_collision",
    primary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    secondary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    fields=("found", "force"),
    reduce="none",
    num_slots=1,
    history_length=4,
  )
  cfg.scene.sensors = (cfg.scene.sensors or ()) + (
    feet_ground_cfg,
    self_collision_cfg,
  )

  if cfg.scene.terrain is not None and cfg.scene.terrain.terrain_generator is not None:
    cfg.scene.terrain.terrain_generator.curriculum = True

  joint_pos_action = cfg.actions["joint_pos"]
  assert isinstance(joint_pos_action, JointPositionActionCfg)
  joint_pos_action.scale = HU_D03_ACTION_SCALE

  cfg.viewer.body_name = TORSO

  twist_cmd = cfg.commands["twist"]
  assert isinstance(twist_cmd, UniformVelocityCommandCfg)
  twist_cmd.viz.z_offset = 1.75

  cfg.events["foot_friction"].params["asset_cfg"].geom_names = FOOT_GEOMS
  cfg.events["base_com"].params["asset_cfg"].body_names = (TORSO,)

  # Same std rationale as the G1: loose knees/hip pitch, tight ankle roll and
  # waist roll/pitch, moderate arms. The HU_D03 also has a head and hand yaw.
  cfg.rewards["pose"].params["std_standing"] = {".*": 0.05}
  cfg.rewards["pose"].params["std_walking"] = {
    # Lower body.
    r".*hip_pitch.*": 0.3,
    r".*hip_roll.*": 0.15,
    r".*hip_yaw.*": 0.15,
    r".*knee.*": 0.35,
    r".*ankle_pitch.*": 0.25,
    r".*ankle_roll.*": 0.1,
    # Waist and head.
    r".*waist_yaw.*": 0.2,
    r".*waist_roll.*": 0.08,
    r".*waist_pitch.*": 0.1,
    r"head_.*": 0.1,
    # Arms.
    r".*shoulder_pitch.*": 0.15,
    r".*shoulder_roll.*": 0.15,
    r".*shoulder_yaw.*": 0.1,
    r".*elbow.*": 0.15,
    r".*wrist.*": 0.3,
    r".*hand_yaw.*": 0.3,
  }
  cfg.rewards["pose"].params["std_running"] = {
    # Lower body.
    r".*hip_pitch.*": 0.5,
    r".*hip_roll.*": 0.2,
    r".*hip_yaw.*": 0.2,
    r".*knee.*": 0.6,
    r".*ankle_pitch.*": 0.35,
    r".*ankle_roll.*": 0.15,
    # Waist and head.
    r".*waist_yaw.*": 0.3,
    r".*waist_roll.*": 0.08,
    r".*waist_pitch.*": 0.2,
    r"head_.*": 0.1,
    # Arms.
    r".*shoulder_pitch.*": 0.5,
    r".*shoulder_roll.*": 0.2,
    r".*shoulder_yaw.*": 0.15,
    r".*elbow.*": 0.35,
    r".*wrist.*": 0.3,
    r".*hand_yaw.*": 0.3,
  }

  cfg.rewards["upright"].params["asset_cfg"].body_names = (TORSO,)
  cfg.rewards["body_ang_vel"].params["asset_cfg"].body_names = (TORSO,)

  for reward_name in ["foot_clearance", "foot_slip"]:
    cfg.rewards[reward_name].params["asset_cfg"].site_names = site_names

  cfg.rewards["body_ang_vel"].weight = -0.05
  cfg.rewards["angular_momentum"].weight = -0.02
  cfg.rewards["air_time"].weight = 0.0

  cfg.rewards["self_collisions"] = RewardTermCfg(
    func=mdp.self_collision_cost,
    weight=-1.0,
    params={"sensor_name": self_collision_cfg.name, "force_threshold": 10.0},
  )

  if play:
    cfg.episode_length_s = int(1e9)

    cfg.observations["actor"].enable_corruption = False
    cfg.events.pop("push_robot", None)
    cfg.terminations.pop("out_of_terrain_bounds", None)
    cfg.curriculum = {}
    cfg.events["randomize_terrain"] = EventTermCfg(
      func=envs_mdp.randomize_terrain,
      mode="reset",
      params={},
    )

    if cfg.scene.terrain is not None:
      if cfg.scene.terrain.terrain_generator is not None:
        cfg.scene.terrain.terrain_generator.curriculum = False
        cfg.scene.terrain.terrain_generator.num_cols = 5
        cfg.scene.terrain.terrain_generator.num_rows = 5
        cfg.scene.terrain.terrain_generator.border_width = 10.0

  return cfg


def hu_d03_flat_env_cfg(play: bool = False) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 flat terrain velocity configuration."""
  cfg = hu_d03_rough_env_cfg(play=play)

  cfg.sim.njmax = 300
  cfg.sim.mujoco.ccd_iterations = 50
  cfg.sim.contact_sensor_maxmatch = 64
  cfg.sim.nconmax = None

  assert cfg.scene.terrain is not None
  cfg.scene.terrain.terrain_type = "plane"
  cfg.scene.terrain.terrain_generator = None

  cfg.scene.sensors = tuple(
    s for s in (cfg.scene.sensors or ()) if s.name != "terrain_scan"
  )
  del cfg.observations["actor"].terms["height_scan"]
  del cfg.observations["critic"].terms["height_scan"]

  cfg.terminations.pop("out_of_terrain_bounds", None)
  cfg.curriculum.pop("terrain_levels", None)

  if play:
    twist_cmd = cfg.commands["twist"]
    assert isinstance(twist_cmd, UniformVelocityCommandCfg)
    twist_cmd.ranges.lin_vel_x = (-1.5, 2.0)
    twist_cmd.ranges.ang_vel_z = (-0.7, 0.7)

  return cfg

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/src/hu_d03_mjlab/tasks/tracking_env_cfgs.py
"""LimX HU_D03 flat tracking (mimic) environment configurations."""

from mjlab.envs import ManagerBasedRlEnvCfg
from mjlab.envs.mdp.actions import JointPositionActionCfg
from mjlab.managers.observation_manager import ObservationGroupCfg
from mjlab.sensor import ContactMatch, ContactSensorCfg
from mjlab.tasks.tracking.mdp import MotionCommandCfg
from mjlab.tasks.tracking.tracking_env_cfg import make_tracking_env_cfg

from hu_d03_mjlab.robot.hu_d03_constants import (
  FOOT_GEOMS,
  HU_D03_ACTION_SCALE,
  get_hu_d03_robot_cfg,
)

PELVIS = "base_link"
TORSO = "waist_pitch_link"

# Bodies whose pose is tracked (same choice of links as the G1 config).
TRACKED_BODIES = (
  PELVIS,
  "left_hip_roll_link",
  "left_knee_link",
  "left_ankle_roll_link",
  "right_hip_roll_link",
  "right_knee_link",
  "right_ankle_roll_link",
  TORSO,
  "left_shoulder_roll_link",
  "left_elbow_link",
  "left_wrist_pitch_link",
  "right_shoulder_roll_link",
  "right_elbow_link",
  "right_wrist_pitch_link",
)


def hu_d03_flat_tracking_env_cfg(
  has_state_estimation: bool = True,
  play: bool = False,
) -> ManagerBasedRlEnvCfg:
  """Create LimX HU_D03 flat terrain tracking configuration."""
  cfg = make_tracking_env_cfg()

  cfg.scene.entities = {"robot": get_hu_d03_robot_cfg()}

  self_collision_cfg = ContactSensorCfg(
    name="self_collision",
    primary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    secondary=ContactMatch(mode="subtree", pattern=PELVIS, entity="robot"),
    fields=("found", "force"),
    reduce="none",
    num_slots=1,
    history_length=4,
  )
  cfg.scene.sensors = (self_collision_cfg,)

  joint_pos_action = cfg.actions["joint_pos"]
  assert isinstance(joint_pos_action, JointPositionActionCfg)
  joint_pos_action.scale = HU_D03_ACTION_SCALE

  motion_cmd = cfg.commands["motion"]
  assert isinstance(motion_cmd, MotionCommandCfg)
  motion_cmd.anchor_body_name = TORSO
  motion_cmd.body_names = TRACKED_BODIES

  cfg.events["foot_friction"].params["asset_cfg"].geom_names = FOOT_GEOMS
  cfg.events["base_com"].params["asset_cfg"].body_names = (TORSO,)

  cfg.terminations["ee_body_pos"].params["body_names"] = (
    "left_ankle_roll_link",
    "right_ankle_roll_link",
    "left_wrist_pitch_link",
    "right_wrist_pitch_link",
  )

  cfg.viewer.body_name = TORSO

  if not has_state_estimation:
    new_actor_terms = {
      k: v
      for k, v in cfg.observations["actor"].terms.items()
      if k not in ["motion_anchor_pos_b", "base_lin_vel"]
    }
    cfg.observations["actor"] = ObservationGroupCfg(
      terms=new_actor_terms,
      concatenate_terms=True,
      enable_corruption=True,
    )

  if play:
    cfg.episode_length_s = int(1e9)

    cfg.observations["actor"].enable_corruption = False
    cfg.events.pop("push_robot", None)

    motion_cmd.pose_range = {}
    motion_cmd.velocity_range = {}

    motion_cmd.sampling_mode = "start"

  return cfg

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/scripts/stand_test.py
"""CPU stand test for the converted HU_D03 model (plain MuJoCo, no GPU, no mjlab).

Adds PD position actuators with the gains from hu_d03_params, puts the robot in
the home pose with its soles on the ground, and holds the pose for a few
seconds. Prints the home base height, any self-contacts at the home pose, and
how far the base drifts.

  python scripts/stand_test.py [--seconds 5] [--viewer]
"""

import argparse
import re
import sys
from pathlib import Path

import mujoco
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from hu_d03_mjlab.robot.hu_d03_params import ACTUATOR_GROUPS, HOME_JOINT_POS  # noqa: E402
from hu_d03_mjlab.robot.hu_d03_spec import get_spec  # noqa: E402


def match(patterns, name):
  return any(re.fullmatch(p, name) for p in patterns)


def build_model():
  spec = get_spec()
  spec.worldbody.add_geom(
    name="floor", type=mujoco.mjtGeom.mjGEOM_PLANE, size=(0, 0, 1),
  )
  spec.option.timestep = 0.005
  for joint in spec.joints:
    if joint.type != mujoco.mjtJoint.mjJNT_HINGE:
      continue
    group = next(g for g in ACTUATOR_GROUPS.values() if match(g.joint_names_expr, joint.name))
    joint.armature = group.armature
    act = spec.add_actuator(name=joint.name, target=joint.name,
                            trntype=mujoco.mjtTrn.mjTRN_JOINT)
    act.set_to_position(kp=group.stiffness, kv=group.damping)
    act.forcelimited = True
    act.forcerange = (-group.effort_limit, group.effort_limit)
  return spec.compile()


def home_qpos(model):
  qpos = model.qpos0.copy()
  for j in range(1, model.njnt):
    name = model.joint(j).name
    for pattern, value in HOME_JOINT_POS.items():
      if re.fullmatch(pattern, name):
        qpos[model.jnt_qposadr[j]] = value
  # Lift or lower the base so the lowest foot site touches z = 0.
  data = mujoco.MjData(model)
  data.qpos[:] = qpos
  data.qpos[2] = 1.0
  mujoco.mj_kinematics(model, data)
  sole = min(data.site("left_foot").xpos[2], data.site("right_foot").xpos[2])
  qpos[2] = 1.0 - sole
  return qpos


def main():
  parser = argparse.ArgumentParser()
  parser.add_argument("--seconds", type=float, default=5.0)
  parser.add_argument("--viewer", action="store_true")
  args = parser.parse_args()

  model = build_model()
  data = mujoco.MjData(model)
  qpos = home_qpos(model)
  data.qpos[:] = qpos
  data.ctrl[:] = qpos[7:]
  mujoco.mj_forward(model, data)

  print(f"home base height: {qpos[2]:.4f} m, total mass {model.body_mass.sum():.2f} kg")
  floor = model.geom("floor").id
  self_contacts = [
    (model.geom(c.geom1).name, model.geom(c.geom2).name)
    for c in data.contact[: data.ncon]
    if floor not in (c.geom1, c.geom2)
  ]
  print(f"self-contacts at home pose: {self_contacts or 'none'}")

  def step():
    mujoco.mj_step(model, data)

  if args.viewer:
    from mujoco import viewer as mj_viewer

    with mj_viewer.launch_passive(model, data) as v:
      while v.is_running() and data.time < args.seconds:
        step()
        v.sync()
  else:
    while data.time < args.seconds:
      step()

  drift = np.linalg.norm(data.qpos[:2] - qpos[:2])
  up = data.xmat[model.body("base_link").id].reshape(3, 3)[:, 2]
  tilt = np.degrees(np.arccos(np.clip(up[2], -1, 1)))
  joint_err = np.abs(data.qpos[7:] - qpos[7:]).max()
  print(f"after {args.seconds:.1f}s: base z {data.qpos[2]:.4f} m, xy drift {drift:.4f} m, "
        f"tilt {tilt:.2f} deg, max joint error {joint_err:.4f} rad")
  ok = data.qpos[2] > qpos[2] - 0.1 and tilt < 10
  print("STAND TEST", "PASSED" if ok else "FAILED")
  return 0 if ok else 1


if __name__ == "__main__":
  sys.exit(main())

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/scripts/retarget_g1_to_hu_d03.py
"""Convert a Unitree G1 motion CSV (LAFAN1 retargeting dataset format) to the HU_D03.

Joint-space transfer, plain MuJoCo on CPU:
  * Joints with the same name are copied. The sign of each joint is chosen by
    comparing the two robots' joint axes in the world frame at the zero pose.
  * The G1 wrist pitch/yaw go to the HU_D03 wrist pitch / hand yaw; the G1
    wrist roll and the HU_D03 head stay at 0.
  * The elbows get a fixed offset because the zero poses differ (ZERO_OFFSETS).
  * Values are clipped to the HU_D03 joint limits.
  * The root height is scaled by the ratio of pelvis heights, then each frame
    is shifted so the feet do not go through the ground.

The HU_D03 is ~1.6 m tall and the G1 ~1.3 m, so foot placement and hand
positions are only approximate. It is good enough to train a tracking policy;
for exact contacts, retarget from the human data with an IK tool instead.

Input CSV (30 fps): x y z qx qy qz qw + 29 G1 joints.
Output CSV (same fps): x y z qx qy qz qw + 31 HU_D03 joints (model order).

  python scripts/retarget_g1_to_hu_d03.py --input walk1_subject1.csv \
      --output walk1_subject1_hu_d03.csv [--g1-xml path/to/g1.xml]
"""

import argparse
import sys
from pathlib import Path

import mujoco
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from hu_d03_mjlab.robot.hu_d03_spec import get_spec  # noqa: E402

G1_JOINTS = (
  "left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint",
  "left_knee_joint", "left_ankle_pitch_joint", "left_ankle_roll_joint",
  "right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint",
  "right_knee_joint", "right_ankle_pitch_joint", "right_ankle_roll_joint",
  "waist_yaw_joint", "waist_roll_joint", "waist_pitch_joint",
  "left_shoulder_pitch_joint", "left_shoulder_roll_joint", "left_shoulder_yaw_joint",
  "left_elbow_joint", "left_wrist_roll_joint", "left_wrist_pitch_joint",
  "left_wrist_yaw_joint",
  "right_shoulder_pitch_joint", "right_shoulder_roll_joint", "right_shoulder_yaw_joint",
  "right_elbow_joint", "right_wrist_roll_joint", "right_wrist_pitch_joint",
  "right_wrist_yaw_joint",
)  # fmt: skip

# HU_D03 joint -> G1 joint, where the names differ.
RENAMED = {
  "left_hand_yaw_joint": "left_wrist_yaw_joint",
  "right_hand_yaw_joint": "right_wrist_yaw_joint",
}
# Zero-pose differences (HU_D03 angle = sign * G1 angle + offset). At zero the
# G1 forearm points forward, the HU_D03 forearm hangs down; found by matching
# the forearm direction of both robots at the G1 zero pose.
ZERO_OFFSETS = {"left_elbow_joint": -1.4984, "right_elbow_joint": -1.4984}
UNMAPPED = {"head_yaw_joint", "head_pitch_joint", "left_wrist_yaw_joint", "right_wrist_yaw_joint"}

HU_FOOT_POINTS = tuple(
  f"contact_{side}_{p}" for side in ("left", "right") for p in ("heel", "center", "tip")
)


def default_g1_xml() -> Path:
  try:
    from mjlab import MJLAB_SRC_PATH

    return MJLAB_SRC_PATH / "asset_zoo/robots/unitree_g1/xmls/g1.xml"
  except ImportError:
    root = Path(__file__).resolve().parents[1]
    return root / "third_party/mjlab/src/mjlab/asset_zoo/robots/unitree_g1/xmls/g1.xml"


def world_axes(model: mujoco.MjModel) -> dict[str, np.ndarray]:
  data = mujoco.MjData(model)
  mujoco.mj_kinematics(model, data)
  mujoco.mj_comPos(model, data)
  return {
    model.joint(j).name: data.xaxis[j].copy()
    for j in range(model.njnt)
    if model.jnt_type[j] == mujoco.mjtJoint.mjJNT_HINGE
  }


def pelvis_height(model: mujoco.MjModel, foot_sites: tuple[str, ...]) -> float:
  data = mujoco.MjData(model)
  data.qpos[2] = 1.0
  mujoco.mj_kinematics(model, data)
  return 1.0 - min(data.site(s).xpos[2] for s in foot_sites)


def build_mapping(g1: mujoco.MjModel, hu: mujoco.MjModel):
  """Returns (hu_qpos_index, g1_column, sign) triples and a printable report."""
  g1_axes, hu_axes = world_axes(g1), world_axes(hu)
  mapping, report = [], []
  for j in range(1, hu.njnt):
    name = hu.joint(j).name
    if name in UNMAPPED:
      report.append(f"  {name:28s} <- 0")
      continue
    src = RENAMED.get(name, name)
    dot = float(np.dot(g1_axes[src], hu_axes[name]))
    if abs(dot) < 0.7:
      report.append(f"  {name:28s} <- 0 (axes differ, dot={dot:+.2f})")
      continue
    sign = 1.0 if dot > 0 else -1.0
    mapping.append((hu.jnt_qposadr[j], G1_JOINTS.index(src), sign))
    report.append(f"  {name:28s} <- {'+' if sign > 0 else '-'}{src} (dot={dot:+.2f})")
  return mapping, report


def retarget(motion: np.ndarray, g1: mujoco.MjModel, hu: mujoco.MjModel, verbose=True):
  mapping, report = build_mapping(g1, hu)
  if verbose:
    print("joint mapping:\n" + "\n".join(report))

  scale = pelvis_height(hu, ("left_foot", "right_foot")) / pelvis_height(
    g1, ("left_foot", "right_foot")
  )
  lo = hu.jnt_range[1:, 0] * 0.98
  hi = hu.jnt_range[1:, 1] * 0.98

  n = motion.shape[0]
  qpos = np.zeros((n, hu.nq))
  qpos[:, :3] = motion[:, :3] * scale
  qpos[:, 3:7] = motion[:, [6, 3, 4, 5]]  # xyzw -> wxyz
  for adr, col, sign in mapping:
    qpos[:, adr] = sign * motion[:, 7 + col]
  for name, offset in ZERO_OFFSETS.items():
    qpos[:, hu.jnt_qposadr[hu.joint(name).id]] += offset
  qpos[:, 7:] = np.clip(qpos[:, 7:], lo, hi)

  # Ground the feet: global shift so the feet touch the ground in most frames,
  # then lift any frame where a foot point is still below the ground.
  data = mujoco.MjData(hu)
  min_z = np.empty(n)
  for i in range(n):
    data.qpos[:] = qpos[i]
    mujoco.mj_kinematics(hu, data)
    min_z[i] = min(data.body(b).xpos[2] for b in HU_FOOT_POINTS)
  qpos[:, 2] -= np.percentile(min_z, 5)
  min_z -= np.percentile(min_z, 5)
  qpos[:, 2] += np.maximum(-min_z, 0.0)

  if verbose:
    print(f"root height scale {scale:.3f}, frames {n}")
  out = np.concatenate([qpos[:, :3], qpos[:, [4, 5, 6, 3]], qpos[:, 7:]], axis=1)
  return out


def main():
  parser = argparse.ArgumentParser(description=__doc__.split("\n")[0])
  parser.add_argument("--input", required=True)
  parser.add_argument("--output", required=True)
  parser.add_argument("--g1-xml", default=None)
  args = parser.parse_args()

  g1 = mujoco.MjModel.from_xml_path(str(args.g1_xml or default_g1_xml()))
  hu = get_spec().compile()
  motion = np.loadtxt(args.input, delimiter=",")
  assert motion.shape[1] == 7 + len(G1_JOINTS), motion.shape
  out = retarget(motion, g1, hu)
  np.savetxt(args.output, out, delimiter=",", fmt="%.6f")
  print(f"wrote {args.output} ({out.shape[0]} frames, {out.shape[1] - 7} joints)")


if __name__ == "__main__":
  main()

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/scripts/csv_to_npz_hu_d03.py
"""Convert an HU_D03 motion CSV to the .npz that mjlab's tracking task reads.

Same as mjlab's scripts/csv_to_npz.py (which hard-codes the G1), but for the
HU_D03, and it writes the .npz to a local path instead of requiring a Weights &
Biases registry upload. Train with:

  train Mjlab-Tracking-Flat-LimX-HU-D03 --env.commands.motion.motion-file motion.npz

  python scripts/csv_to_npz_hu_d03.py --input-file walk_hu_d03.csv \
      --output-file walk_hu_d03.npz [--input-fps 30] [--output-fps 50]
"""

from pathlib import Path

import numpy as np
import torch
import tyro
from tqdm import tqdm

import mjlab  # noqa: F401  (configures Warp, loads the HU_D03 tasks)
from mjlab.entity import Entity
from mjlab.scene import Scene
from mjlab.scripts.csv_to_npz import MotionLoader
from mjlab.sim.sim import Simulation, SimulationCfg

from hu_d03_mjlab.tasks.tracking_env_cfgs import hu_d03_flat_tracking_env_cfg

LOG_KEYS = (
  "joint_pos",
  "joint_vel",
  "body_pos_w",
  "body_quat_w",
  "body_lin_vel_w",
  "body_ang_vel_w",
)


def main(
  input_file: str,
  output_file: str,
  input_fps: float = 30.0,
  output_fps: float = 50.0,
  device: str = "cuda:0",
  line_range: tuple[int, int] | None = None,
):
  """Args:
    line_range: Only convert these CSV lines (1-based, inclusive), e.g. (1, 1500).
  """
  if device.startswith("cuda") and not torch.cuda.is_available():
    print("[WARNING]: CUDA is not available. Falling back to CPU.")
    device = "cpu"

  sim_cfg = SimulationCfg()
  sim_cfg.mujoco.timestep = 1.0 / output_fps

  scene = Scene(hu_d03_flat_tracking_env_cfg().scene, device=device)
  model = scene.compile()
  sim = Simulation(num_envs=1, cfg=sim_cfg, model=model, device=device)
  scene.initialize(sim.mj_model, sim.model, sim.data)

  robot: Entity = scene["robot"]
  # CSV columns are the HU_D03 joints in model order.
  joint_names = list(robot.joint_names)

  motion = MotionLoader(
    motion_file=input_file,
    input_fps=int(input_fps),
    output_fps=int(output_fps),
    device=sim.device,
    line_range=line_range,
  )
  assert motion.motion_dof_poss_input.shape[1] == len(joint_names), (
    f"CSV has {motion.motion_dof_poss_input.shape[1]} joints, robot has "
    f"{len(joint_names)}"
  )
  joint_ids = robot.find_joints(joint_names, preserve_order=True)[0]

  log: dict[str, list] = {k: [] for k in LOG_KEYS}
  scene.reset()

  for _ in tqdm(range(motion.output_frames), desc="frames"):
    (
      (base_pos, base_rot, base_lin_vel, base_ang_vel, dof_pos, dof_vel),
      _,
    ) = motion.get_next_state()

    root_states = robot.data.default_root_state.clone()
    root_states[:, 0:3] = base_pos
    root_states[:, :2] += scene.env_origins[:, :2]
    root_states[:, 3:7] = base_rot
    root_states[:, 7:10] = base_lin_vel
    root_states[:, 10:] = base_ang_vel
    robot.write_root_state_to_sim(root_states)

    joint_pos = robot.data.default_joint_pos.clone()
    joint_vel = robot.data.default_joint_vel.clone()
    joint_pos[:, joint_ids] = dof_pos
    joint_vel[:, joint_ids] = dof_vel
    robot.write_joint_state_to_sim(joint_pos, joint_vel)

    sim.forward()
    scene.update(sim.mj_model.opt.timestep)

    data = robot.data
    log["joint_pos"].append(data.joint_pos[0].cpu().numpy().copy())
    log["joint_vel"].append(data.joint_vel[0].cpu().numpy().copy())
    log["body_pos_w"].append(data.body_link_pos_w[0].cpu().numpy().copy())
    log["body_quat_w"].append(data.body_link_quat_w[0].cpu().numpy().copy())
    log["body_lin_vel_w"].append(data.body_link_lin_vel_w[0].cpu().numpy().copy())
    log["body_ang_vel_w"].append(data.body_link_ang_vel_w[0].cpu().numpy().copy())

  out = {"fps": [output_fps], **{k: np.stack(v, axis=0) for k, v in log.items()}}
  Path(output_file).parent.mkdir(parents=True, exist_ok=True)
  np.savez(output_file, **out)
  print(f"wrote {output_file} ({len(log['joint_pos'])} frames at {output_fps} fps)")


if __name__ == "__main__":
  tyro.cli(main, config=mjlab.TYRO_FLAGS)

In [ ]:
%%writefile /tmp/hu_d03_code/hu-d03-rl/scripts/record_video.py
"""Roll out a policy headless and save an .mp4 (for Colab/Kaggle, where mjlab's
`play` viewer cannot open a window).

  python scripts/record_video.py Mjlab-Velocity-Flat-LimX-HU-D03 \
      --checkpoint-file logs/.../model_5000.pt --output-file walk.mp4
  python scripts/record_video.py Mjlab-Tracking-Flat-LimX-HU-D03 \
      --checkpoint-file ... --motion-file motion.npz --output-file mimic.mp4
  # --agent zero holds the default pose (stand test, no checkpoint needed).
  # Defaults to 1280x720; e.g. --width 1920 --height 1080 --distance 4 for 1080p.
"""

from dataclasses import asdict
from pathlib import Path
from typing import Literal

import mediapy
import torch
import tyro

import mjlab
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg, load_runner_cls
from mjlab.tasks.tracking.mdp import MotionCommandCfg


def main(
  task_id: tyro.conf.Positional[str],
  output_file: str,
  checkpoint_file: str | None = None,
  motion_file: str | None = None,
  agent: Literal["trained", "zero"] = "trained",
  seconds: float = 10.0,
  no_terminations: bool = False,
  width: int = 1280,
  height: int = 720,
  distance: float = 3.0,
  elevation: float = -15.0,
  azimuth: float = 90.0,
):
  device = "cuda:0" if torch.cuda.is_available() else "cpu"
  env_cfg = load_env_cfg(task_id, play=True)
  agent_cfg = load_rl_cfg(task_id)
  env_cfg.scene.num_envs = 1
  # mjlab renders 320x240 from 5 m away by default; frame the whole robot instead.
  env_cfg.viewer.width, env_cfg.viewer.height = width, height
  env_cfg.viewer.distance = distance
  env_cfg.viewer.elevation = elevation
  env_cfg.viewer.azimuth = azimuth
  if no_terminations:
    env_cfg.terminations = {}

  motion_cmd = env_cfg.commands.get("motion")
  if isinstance(motion_cmd, MotionCommandCfg):
    if motion_file is None:
      raise ValueError("Tracking tasks need --motion-file.")
    motion_cmd.motion_file = motion_file

  env = ManagerBasedRlEnv(cfg=env_cfg, device=device, render_mode="rgb_array")
  env = RslRlVecEnvWrapper(env, clip_actions=agent_cfg.clip_actions)

  if agent == "zero":
    shape = env.unwrapped.action_space.shape

    def policy(obs):
      del obs
      return torch.zeros(shape, device=env.unwrapped.device)
  else:
    if checkpoint_file is None:
      raise ValueError("--checkpoint-file is required with --agent trained.")
    runner_cls = load_runner_cls(task_id) or MjlabOnPolicyRunner
    runner = runner_cls(env, asdict(agent_cfg), device=device)
    runner.load(checkpoint_file, load_cfg={"actor": True}, strict=True,
                map_location=device)
    policy = runner.get_inference_policy(device=device)

  obs = env.get_observations()
  frames = []
  steps = int(seconds / env.unwrapped.step_dt)
  with torch.inference_mode():
    for _ in range(steps):
      obs, _, _, _ = env.step(policy(obs))
      frames.append(env.unwrapped.render())

  fps = round(1.0 / env.unwrapped.step_dt)
  Path(output_file).parent.mkdir(parents=True, exist_ok=True)
  mediapy.write_video(output_file, frames, fps=fps)
  print(f"wrote {output_file} ({len(frames)} frames at {fps} fps)")
  env.close()


if __name__ == "__main__":
  tyro.cli(main, config=mjlab.TYRO_FLAGS)

In [ ]:
sh(f"uv pip install --system -q -e {CODE / 'hu-d03-rl'}")

## 4. Sanity checks

* The stand test holds the home pose with the PD gains in plain MuJoCo (CPU).
* `list-envs` should show the four `LimX-HU-D03` tasks.
* The zero-action video runs the full mjlab environment on the GPU. The robot should stand still.

In [ ]:
SCRIPTS = CODE / "hu-d03-rl" / "scripts"
sh(f"python {SCRIPTS / 'stand_test.py'} --seconds 10")
sh("list-envs | grep -i hu-d03")

In [ ]:
from IPython.display import Video

def video(task, out, *extra):
  sh(f"cd {WORK} && python {SCRIPTS / 'record_video.py'} {task} --output-file {out} " + " ".join(extra))
  return Video(str(WORK / out), embed=True, width=640)

video("Mjlab-Velocity-Flat-LimX-HU-D03", "videos/zero_action.mp4", "--agent zero", "--seconds 5")

If rendering fails with an EGL error (seen on some Kaggle images), run this cell and then the video cell again:

In [ ]:
sh("apt-get -qq install -y libosmesa6 > /dev/null")
os.environ["MUJOCO_GL"] = "osmesa"

## 5. Train the locomotion policy

Logs and checkpoints go to `WORK/logs/rsl_rl/hu_d03_velocity/<date_time>/`.
Each save also writes an ONNX copy of the policy for deployment.

**Resume after a disconnect:** re-run sections 1–3 (skip the checks), set `RESUME = True` and run this cell again.
It continues from the newest checkpoint of the newest run.

In [ ]:
RESUME = False

def train(task, iterations, *extra):
  gpus = '--gpu-ids "[0, 1]"' if USE_TWO_GPUS and NUM_GPUS >= 2 else ""
  resume = "--agent.resume True" if RESUME else ""
  sh(f"cd {WORK} && train {task} --env.scene.num-envs {NUM_ENVS} "
     f"--agent.max-iterations {iterations} --agent.logger tensorboard {gpus} {resume} " + " ".join(extra))

train("Mjlab-Velocity-Flat-LimX-HU-D03", VELOCITY_ITERATIONS)

### Watch the training curves

On Colab, run this in a separate cell while training (or after). Look at `Episode_Reward/track_linear_velocity`
and the mean episode length: the robot is walking when the episode length reaches the 20 s limit.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {WORK}/logs/rsl_rl

### Video of the trained locomotion policy

In [ ]:
def latest_checkpoint(experiment):
  runs = sorted((WORK / "logs" / "rsl_rl" / experiment).glob("*/"))
  ckpts = sorted(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
  print("checkpoint:", ckpts[-1])
  return ckpts[-1]

video("Mjlab-Velocity-Flat-LimX-HU-D03", "videos/locomotion.mp4",
      f"--checkpoint-file {latest_checkpoint('hu_d03_velocity')}", "--seconds 10")

## 6. Prepare mimic data

[GMR](https://github.com/YanjieZe/GMR) and similar retargeting tools do not support the HU_D03 yet, so this
notebook starts from the LAFAN1 motions that Unitree already retargeted to the G1
([lvhaidong/LAFAN1_Retargeting_Dataset](https://huggingface.co/datasets/lvhaidong/LAFAN1_Retargeting_Dataset))
and copies them to the HU_D03 joint by joint (`scripts/retarget_g1_to_hu_d03.py`): joint signs are matched by
comparing joint axes, the elbows get a zero-pose offset, and the feet are kept above the ground.

Other clips: `dance1_subject1..3`, `dance2_subject1..5`, `run1_subject2`, `jumps1_subject1`, `fight1_subject2`, ...
LAFAN1 is licensed CC BY-NC-ND 4.0 (non-commercial).

In [ ]:
MOTION = "walk1_subject1"
LINE_RANGE = None  # e.g. (1, 1800) to use only the first 60 s of a long clip.

data = WORK / "motions"
data.mkdir(exist_ok=True)
url = f"https://huggingface.co/datasets/lvhaidong/LAFAN1_Retargeting_Dataset/resolve/main/g1/{MOTION}.csv"
if not (data / f"{MOTION}_g1.csv").exists():
  sh(f"wget -q -O {data / f'{MOTION}_g1.csv'} {url}")
sh(f"python {SCRIPTS / 'retarget_g1_to_hu_d03.py'} --input {data / f'{MOTION}_g1.csv'} "
   f"--output {data / f'{MOTION}_hu_d03.csv'} "
   f"--g1-xml {CODE / 'mjlab/src/mjlab/asset_zoo/robots/unitree_g1/xmls/g1.xml'}")
lr = f'--line-range "({LINE_RANGE[0]}, {LINE_RANGE[1]})"' if LINE_RANGE else ""
sh(f"python {SCRIPTS / 'csv_to_npz_hu_d03.py'} --input-file {data / f'{MOTION}_hu_d03.csv'} "
   f"--output-file {data / f'{MOTION}_hu_d03.npz'} {lr}")
MOTION_FILE = data / f"{MOTION}_hu_d03.npz"

Replay the reference motion (no policy, terminations off) to check the retargeting looks right:

In [ ]:
video("Mjlab-Tracking-Flat-LimX-HU-D03", f"videos/{MOTION}_reference.mp4",
      f"--motion-file {MOTION_FILE}", "--agent zero", "--no-terminations True", "--seconds 10")

## 7. Train the mimic policy

Same resume rule as above: set `RESUME = True` to continue the newest `hu_d03_tracking` run.

In [ ]:
RESUME = False
train("Mjlab-Tracking-Flat-LimX-HU-D03", TRACKING_ITERATIONS,
      f"--env.commands.motion.motion-file {MOTION_FILE}")

In [ ]:
video("Mjlab-Tracking-Flat-LimX-HU-D03", f"videos/{MOTION}_mimic.mp4",
      f"--checkpoint-file {latest_checkpoint('hu_d03_tracking')}",
      f"--motion-file {MOTION_FILE}", "--seconds 15")

## 8. Download the results

Zips the newest run of each task (checkpoints, ONNX policies, configs, videos).
On Colab the files are also already in your Drive under `hu_d03_rl/`.

In [ ]:
import shutil
for experiment in ["hu_d03_velocity", "hu_d03_tracking"]:
  runs = sorted((WORK / "logs" / "rsl_rl" / experiment).glob("*/"))
  if runs:
    archive = shutil.make_archive(str(WORK / f"{experiment}_{runs[-1].name}"), "zip", runs[-1])
    print("wrote", archive)
    if ON_COLAB:
      from google.colab import files
      files.download(archive)